In [4]:
"""
FLOCMOD aggregation gain and loss terms
========================================
Implements equations (11)–(13) from Penaloza-Giraldo et al. (2024),
which faithfully restate Verney et al. (2011).

Notation follows the paper exactly:
  k, i, j  -- size-class indices  (0-based here, 1-based in the paper)
  nf        -- number concentration vector  [m^-3],  shape (K,)
  df        -- floc diameter array          [m],      shape (K,)
  S         -- turbulent shear rate         [s^-1],   scalar  (= G in Smoluchowski)
  alpha     -- collision efficiency         [-],      scalar (sticking probability)
  Nf        -- total number of size classes

Aggregation collision kernel (rectilinear / orthokinetic, shear-dominated):
    A(i, j) = (1/6) * S * (df[i] + df[j])^3          (Eq. 13)

Aggregation GAIN for class k  (Eq. 11):
    G_ag(k) = (1/2) * alpha * sum_{i+j -> k} A(i,j) * nf[i] * nf[j]

    The sum is over all pairs (i, j) such that colliding flocs i and j
    produce a floc whose primary-particle count equals that of class k.
    In a discrete size-class model this is implemented by finding the
    *nearest* class index for the combined volume (see _nearest_class).

Aggregation LOSS for class k  (Eq. 12):
    L_ag(k) = alpha * nf[k] * sum_{i=1}^{Nf} A(i, k) * nf[i]

    Every collision that removes a floc from class k, regardless of where
    the daughter product ends up.

Mass conservation check:
    sum_k  nf[k] * p[k] * mp  = const
    where p[k] = (df[k]/dp)^fd  is the number of primary particles per floc
    and   mp   = rho_p * pi/6 * dp^3.

References
----------
Verney R., Lafite R., Brun-Cottan J.-C., Le Hir P. (2011).
  Behaviour of a floc population during a tidal cycle.
  Continental Shelf Research, 31(10), S64–S83.

Penaloza-Giraldo J.A., Hsu T.-J., Manning A.J., et al. (2024).
  A modeling framework for flocculated cohesive sediment transport.
  Advances in Water Resources.  (Eqs. 10–13, 16–19)
"""

import numpy as np


# ---------------------------------------------------------------------------
# Grid setup helpers
# ---------------------------------------------------------------------------

def build_size_grid(dp: float, d_max: float, n_classes: int) -> np.ndarray:
    """
    Return log-spaced floc diameter array [m].

    Parameters
    ----------
    dp        : primary particle diameter [m]
    d_max     : maximum floc diameter [m]
    n_classes : number of size classes K
    """
    return np.geomspace(dp, d_max, n_classes)


def primary_particle_count(df: np.ndarray, dp: float, fd: float) -> np.ndarray:
    """
    Number of primary particles inside each floc class (fractal scaling).

    p(k) = (df[k] / dp)^fd                     Kranenburg (1994)

    Parameters
    ----------
    df  : floc diameter array [m], shape (K,)
    dp  : primary particle diameter [m]
    fd  : fractal dimension [-], typically ~2.1

    Returns
    -------
    p   : shape (K,), dimensionless (may be non-integer)
    """
    return (df / dp) ** fd


def floc_density(df: np.ndarray, dp: float, fd: float,
                 rho_p: float, rho_w: float) -> np.ndarray:
    """
    Floc density via fractal relation (Kranenburg 1994, Eq. 18).

    rho_f(k) = rho_w + (rho_p - rho_w) * (dp / df[k])^(3 - fd)
    """
    return rho_w + (rho_p - rho_w) * (dp / df) ** (3.0 - fd)


# ---------------------------------------------------------------------------
# Nearest-class mapping for aggregation gain
# ---------------------------------------------------------------------------

def _build_aggregation_map(df: np.ndarray, dp: float, fd: float) -> dict:
    """
    Pre-compute which output class k receives mass when classes i and j collide.

    Strategy: the combined floc has primary-particle count
        p_combined = p[i] + p[j]
    We map this to the nearest class by p-value (log-distance).

    Returns
    -------
    agg_map : dict  {(i, j): k}   for i <= j
    """
    p = primary_particle_count(df, dp, fd)
    K = len(df)
    agg_map = {}
    for i in range(K):
        for j in range(i, K):          # i <= j  (symmetry handled below)
            p_combined = p[i] + p[j]
            if p_combined > p[-1]:     # combined size exceeds grid -- cap at max class
                k_out = K - 1
            else:
                # nearest class in log(p) space
                k_out = int(np.argmin(np.abs(np.log(p) - np.log(p_combined))))
            agg_map[(i, j)] = k_out
    return agg_map


# ---------------------------------------------------------------------------
# Collision kernel
# ---------------------------------------------------------------------------

def collision_kernel(df: np.ndarray, S: float) -> np.ndarray:
    """
    Rectilinear (orthokinetic) shear collision kernel matrix.

    A(i, j) = (1/6) * S * (df[i] + df[j])^3       (Eq. 13)

    Parameters
    ----------
    df  : floc diameter array [m], shape (K,)
    S   : turbulent shear rate [s^-1]

    Returns
    -------
    A   : shape (K, K),  units [m^3 s^-1]
    """
    d_sum = df[:, None] + df[None, :]          # (K, K)  broadcasting
    return (S / 6.0) * d_sum ** 3


# ---------------------------------------------------------------------------
# Aggregation gain and loss
# ---------------------------------------------------------------------------

def aggregation_gain(nf: np.ndarray, A: np.ndarray,
                     alpha: float, agg_map: dict) -> np.ndarray:
    """
    Aggregation gain term G_ag(k) for every class k.

    G_ag(k) = (1/2) * alpha * sum_{(i,j)->k}  A(i,j) * nf[i] * nf[j]

    The factor 1/2 prevents double-counting: the sum over all ordered
    pairs (i,j) with i<=j already counts each collision once; if we
    iterated over all (i,j) and (j,i) we would need the 1/2.

    Parameters
    ----------
    nf      : number concentration [m^-3],  shape (K,)
    A       : collision kernel [m^3 s^-1],  shape (K, K)
    alpha   : collision efficiency [-]
    agg_map : dict {(i,j): k_out} from _build_aggregation_map

    Returns
    -------
    G_ag    : shape (K,),  units [m^-3 s^-1]
    """
    K = len(nf)
    G_ag = np.zeros(K)
    for (i, j), k_out in agg_map.items():
        rate = alpha * A[i, j] * nf[i] * nf[j]
        if i == j:
            # pair (i,i): only one distinct ordered pair, but the
            # 1/2 factor still applies -- net coefficient is 1/2.
            G_ag[k_out] += 0.5 * rate
        else:
            # pair (i,j) with i<j: accounts for both (i,j) and (j,i)
            # combined, so coefficient is 1 (= 2 * 1/2).
            G_ag[k_out] += rate
    return G_ag


def aggregation_loss(nf: np.ndarray, A: np.ndarray, alpha: float) -> np.ndarray:
    """
    Aggregation loss term L_ag(k) for every class k.

    L_ag(k) = alpha * nf[k] * sum_{i=0}^{K-1}  A(i, k) * nf[i]

    Every collision involving a floc of class k removes it from class k,
    regardless of where the merged product lands.

    Parameters
    ----------
    nf    : number concentration [m^-3],  shape (K,)
    A     : collision kernel [m^3 s^-1],  shape (K, K)
    alpha : collision efficiency [-]

    Returns
    -------
    L_ag  : shape (K,),  units [m^-3 s^-1]
    """
    # A[i, k] * nf[i]  summed over i  ==  A[:, k] @ nf
    return alpha * nf * (A @ nf)


# ---------------------------------------------------------------------------
# Combined aggregation tendency
# ---------------------------------------------------------------------------

def aggregation_tendency(nf: np.ndarray, df: np.ndarray, S: float,
                         alpha: float, dp: float, fd: float,
                         agg_map: dict | None = None) -> np.ndarray:
    """
    Net aggregation tendency dN/dt|_agg = G_ag - L_ag  for all classes.

    Parameters
    ----------
    nf      : number concentration [m^-3],  shape (K,)
    df      : floc diameters [m],            shape (K,)
    S       : turbulent shear rate [s^-1]
    alpha   : collision efficiency [-]
    dp      : primary particle diameter [m]
    fd      : fractal dimension [-]
    agg_map : pre-computed map (built once; pass None to build on the fly)

    Returns
    -------
    dN_agg  : shape (K,),  [m^-3 s^-1]
    """
    if agg_map is None:
        agg_map = _build_aggregation_map(df, dp, fd)
    A = collision_kernel(df, S)
    G = aggregation_gain(nf, A, alpha, agg_map)
    L = aggregation_loss(nf, A, alpha)
    return G - L


# ---------------------------------------------------------------------------
# Mass-conservation diagnostic
# ---------------------------------------------------------------------------

def total_primary_particle_mass(nf: np.ndarray, df: np.ndarray,
                                dp: float, fd: float, rho_p: float) -> float:
    """
    Total suspended sediment mass concentration [kg m^-3].

    C = sum_k  nf[k] * p[k] * mp
    where  mp = rho_p * pi/6 * dp^3
    """
    p  = primary_particle_count(df, dp, fd)
    mp = rho_p * (np.pi / 6.0) * dp ** 3
    return float(np.sum(nf * p) * mp)


# ---------------------------------------------------------------------------
# Quick smoke-test
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # --- parameters (Verney 2011 / Sherwood 2018 defaults) ---
    dp    = 4e-6          # primary particle diameter [m]
    d_max = 1500e-6       # maximum floc diameter [m]
    K     = 15            # number of size classes
    fd    = 1.9           # fractal dimension
    rho_p = 2650.0        # primary particle density [kg m^-3]
    rho_w = 1025.0        # water density [kg m^-3]
    alpha = 0.4           # collision efficiency
    S     = 1.0           # shear rate [s^-1]

    df = build_size_grid(dp, d_max, K)
    print("Floc diameters [µm]:", np.round(df * 1e6, 1))

    # initial number concentration: uniform mass of 0.093 kg/m^3 in smallest class
    p  = primary_particle_count(df, dp, fd)
    mp = rho_p * (np.pi / 6.0) * dp ** 3
    C0 = 0.093            # [kg m^-3]
    nf = np.zeros(K)
    nf[0] = C0 / (p[0] * mp)   # all mass in primary-particle class
    print(f"\nInitial total mass: {total_primary_particle_mass(nf, df, dp, fd, rho_p):.4f} kg/m^3")

    # pre-compute mapping (do this once before time-stepping)
    agg_map = _build_aggregation_map(df, dp, fd)

    # evaluate tendencies
    A    = collision_kernel(df, S)
    G_ag = aggregation_gain(nf, A, alpha, agg_map)
    L_ag = aggregation_loss(nf, A, alpha)
    dN   = G_ag - L_ag

    print("\nG_ag (first 5 classes) [m^-3 s^-1]:", G_ag[:5])
    print("L_ag (first 5 classes) [m^-3 s^-1]:", L_ag[:5])
    print("dN   (first 5 classes) [m^-3 s^-1]:", dN[:5])

    # mass change implied by one Euler step (should be ~0 for aggregation alone)
    dt    = 1.0           # [s]
    nf_new = np.maximum(nf + dN * dt, 0.0)
    C_new = total_primary_particle_mass(nf_new, df, dp, fd, rho_p)
    print(f"\nMass after one Euler step (dt={dt}s): {C_new:.4f} kg/m^3")
    print("Mass change:", C_new - C0, "(should be near zero for pure aggregation)")

Floc diameters [µm]: [   4.     6.1    9.3   14.2   21.8   33.2   50.7   77.5  118.3  180.6
  275.8  421.2  643.2  982.3 1500. ]

Initial total mass: 0.0930 kg/m^3

G_ag (first 5 classes) [m^-3 s^-1]: [       0.         18718268.08093798        0.                0.
        0.        ]
L_ag (first 5 classes) [m^-3 s^-1]: [37436536.16187597        0.                0.                0.
        0.        ]
dN   (first 5 classes) [m^-3 s^-1]: [-37436536.16187597  18718268.08093798         0.
         0.                 0.        ]

Mass after one Euler step (dt=1.0s): 0.0930 kg/m^3
Mass change: 3.910961399872237e-07 (should be near zero for pure aggregation)
